# GeoLife EDA — Checkpoint 1

Goal: understand raw GeoLife GPS data before choosing cleaning or stay-point thresholds. We will verify dataset structure, profile user/trajectory imbalance, sampling rate, temporal/spatial coverage, movement/noise, altitude quality, transport-label coverage, and privacy risks.

Important source sanity check: the v1.3 guide has an internal inconsistency — its page-1 narrative says 17,621 trajectories, while the v1.3 comparison table says 18,670 trajectories and 24,876,978 points. The actual files are the measurement source of truth.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from notebooks.eda_core import inventory_dataset, read_plt, summarize_all, read_labels

pd.set_option('display.max_columns', 50)
DATA_ROOT = Path(os.environ.get('GEOLIFE_DATA_ROOT', '/data/Geolife Trajectories 1.3/Data'))
print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT.exists(), 'Set GEOLIFE_DATA_ROOT to the folder containing user directories 000..181'

## 1. Dataset inventory and user imbalance
Count users, trajectory files, and labeled users without loading every GPS point. Then inspect the long tail of trajectories per user.

In [ ]:
inventory, all_files = inventory_dataset(DATA_ROOT)
display(inventory.head())
print('users:', len(inventory))
print('trajectory files:', int(inventory.trajectory_count.sum()))
print('users with labels:', int(inventory.has_labels.sum()))
display(inventory.trajectory_count.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
fig, ax = plt.subplots(figsize=(10,4))
inventory.trajectory_count.sort_values().reset_index(drop=True).plot(ax=ax)
ax.set(title='Trajectory count per user (sorted)', xlabel='User rank', ylabel='Trajectory count'); plt.show()

## 2. Raw schema spot-check
PLT files have six header rows. The guide documents latitude, longitude, altitude, serial date, date, and time; altitude -777 is invalid; timestamps are GMT/UTC.

In [ ]:
example_user, example_path = all_files[0]
example = read_plt(example_path)
print(example_user, example_path)
display(example.head())
display(example.dtypes)
display(example[['latitude','longitude','altitude_ft','timestamp']].describe(include='all'))

## 3. Full trajectory-level scan
Do not concatenate ~25M raw points into one DataFrame. Reduce each trajectory to one row: point count, time span, approximate distance, sampling interval, speed diagnostics, coordinate validity, duplicate timestamps, and altitude missingness.

In [ ]:
traj = summarize_all(all_files)
print('trajectory summary shape:', traj.shape)
display(traj.head())
cols = ['n_points','duration_min','distance_km','median_sampling_s','p95_sampling_s','max_speed_kmh','p99_speed_kmh','missing_altitude_rate']
display(traj[cols].describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]))
print('invalid lat:', int(traj.invalid_lat.sum()))
print('invalid lon:', int(traj.invalid_lon.sum()))
print('null timestamp:', int(traj.null_timestamp.sum()))
print('duplicate timestamp:', int(traj.duplicate_timestamp.sum()))
print('non-monotonic trajectories:', int(traj.non_monotonic_timestamp.sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
traj.median_sampling_s.clip(upper=120).dropna().plot(kind='hist', bins=80, ax=ax)
ax.set(title='Median sampling interval per trajectory (clipped at 120s)', xlabel='seconds'); plt.show()
fig, ax = plt.subplots(figsize=(8,4))
traj.duration_min.clip(upper=24*60).dropna().plot(kind='hist', bins=80, ax=ax)
ax.set(title='Trajectory duration (clipped at 24h)', xlabel='minutes'); plt.show()
fig, ax = plt.subplots(figsize=(8,4))
traj.distance_km.clip(upper=200).dropna().plot(kind='hist', bins=80, ax=ax)
ax.set(title='Trajectory distance (clipped at 200km)', xlabel='km'); plt.show()

## 4. Per-user temporal coverage
Home/Office inference needs repeated history. A user observed for a few days is qualitatively different from one observed for months.

In [ ]:
user_summary = traj.groupby('user_id').agg(trajectory_count=('trajectory_id','count'), point_count=('n_points','sum'), first_time=('start_time','min'), last_time=('end_time','max'), total_distance_km=('distance_km','sum'), total_duration_min=('duration_min','sum')).reset_index()
user_summary['calendar_span_days'] = (user_summary.last_time-user_summary.first_time).dt.total_seconds()/86400
display(user_summary.describe(include='all'))
fig, ax = plt.subplots(figsize=(8,4))
user_summary.calendar_span_days.clip(upper=800).plot(kind='hist', bins=60, ax=ax)
ax.set(title='Calendar collection span per user', xlabel='days'); plt.show()

## 5. Movement/noise diagnostics before filtering
Do not pick a speed threshold yet. First inspect extreme segment speeds, duplicated timestamps, and long sampling gaps. Distance alone does not imply noise; speed depends on both distance and time delta.

In [ ]:
display(traj[['user_id','trajectory_id','max_speed_kmh','p99_speed_kmh','median_sampling_s','p95_sampling_s','distance_km','duration_min']].sort_values('max_speed_kmh', ascending=False).head(30))
for threshold in [100,150,200,300,500,1000]:
    print(f'> {threshold:4d} km/h:', int((traj.max_speed_kmh > threshold).sum()), 'trajectories')
display(traj.missing_altitude_rate.describe(percentiles=[.25,.5,.75,.9,.95,.99]))

## 6. Privacy-safe spatial sample
Start with an aggregate/coarse density view. Avoid committing individual users' full trajectory maps because mobility histories can reveal sensitive routines.

In [ ]:
rng = np.random.default_rng(42)
idx = rng.choice(len(all_files), size=min(300, len(all_files)), replace=False)
samples = []
for i in idx:
    user_id, path = all_files[i]
    df = read_plt(path)
    if len(df) > 200:
        df = df.iloc[np.linspace(0, len(df)-1, 200, dtype=int)]
    samples.append(df[['latitude','longitude','timestamp']].assign(user_id=user_id))
points_sample = pd.concat(samples, ignore_index=True)
display(points_sample[['latitude','longitude']].describe())
fig, ax = plt.subplots(figsize=(8,6))
hb = ax.hexbin(points_sample.longitude, points_sample.latitude, gridsize=100, mincnt=1, bins='log')
ax.set(title='Coarse spatial density from reproducible sample', xlabel='longitude', ylabel='latitude')
fig.colorbar(hb, ax=ax, label='log count'); plt.show()

## 7. Transportation-label coverage
These labels describe movement mode, not Home/Office. They are secondary signals for EDA and future movement diagnostics.

In [ ]:
label_frames = []
for user_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.isdigit()):
    path = user_dir / 'labels.txt'
    if path.exists(): label_frames.append(read_labels(path, user_dir.name))
labels = pd.concat(label_frames, ignore_index=True) if label_frames else pd.DataFrame()
print('label rows:', len(labels), 'label users:', labels.user_id.nunique() if not labels.empty else 0)
if not labels.empty:
    labels['duration_min'] = (labels.end_time-labels.start_time).dt.total_seconds()/60
    display(labels['mode'].value_counts())
    display(labels.groupby('mode').duration_min.agg(['count','sum','median']).sort_values('sum', ascending=False))

## 8. Critical DS issue: UTC vs local behavioral time
The guide says PLT timestamps are GMT/UTC. Rules such as `night = Home` and `09:00–17:00 = Office` require an explicit timezone policy. GeoLife contains data outside Beijing, so blindly adding +8 hours everywhere can be wrong. This decision must be documented before implementing the heuristic.

In [ ]:
points_sample['hour_utc'] = points_sample.timestamp.dt.hour
fig, ax = plt.subplots(figsize=(9,4))
points_sample.hour_utc.value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set(title='Sampled GPS points by hour (UTC)', xlabel='hour', ylabel='sampled points'); plt.show()

## 9. Persist local summaries and write conclusions
Generated data is ignored by Git. Review privacy and dataset-license implications before publishing derived artifacts.

After execution, record: observed users/trajectories/points, labeled users, time span, sampling distribution, major quality issues, suspicious speed behavior, altitude usefulness, spatial concentration, user-history imbalance, and the evidence needed for stay-point threshold candidates.

In [ ]:
REPORT_DIR = Path(os.environ.get('GEOLIFE_EDA_OUTPUT', 'reports/eda/generated'))
REPORT_DIR.mkdir(parents=True, exist_ok=True)
inventory.to_csv(REPORT_DIR/'dataset_inventory.csv', index=False)
traj.to_parquet(REPORT_DIR/'trajectory_summary.parquet', index=False)
user_summary.to_csv(REPORT_DIR/'user_summary.csv', index=False)
if not labels.empty:
    labels.groupby('mode').agg(label_count=('mode','size'), duration_min=('duration_min','sum')).reset_index().to_csv(REPORT_DIR/'transport_mode_summary.csv', index=False)
print('wrote local EDA summaries to', REPORT_DIR.resolve())